Import Required Libraries

In [1]:
# Basic data handling
import pandas as pd
import numpy as np

# Train-test splitting
from sklearn.model_selection import train_test_split

# Preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Model
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load Dataset

In [2]:
# Change this to your actual file path
# Reading your CSV into a DataFrame.
file_path = "data/SLIIT_IT_Student_Data_Template_filled-updated.csv"

df = pd.read_csv(file_path)

print("First 5 rows:")
print(df.head())

print("\nColumns:")
print(df.columns)


First 5 rows:
   Gender District_Location Education_Stream  \
0    Male           Colombo       Bio Stream   
1  Female           Gampaha     Maths Stream   
2    Male             Kandy       Bio Stream   
3  Female             Galle     Maths Stream   
4    Male            Matara  Commerce Stream   

                                         Soft_Skills  \
0           Communication, Teamwork, Time Management   
1     Leadership, Problem Solving, Critical Thinking   
2            Public Speaking, Teamwork, Adaptability   
3  Communication, Presentation Skills, Emotional ...   
4   Time Management, Work Ethic, Attention to Detail   

                                           Key_Skils Current_semester  GPA  \
0  Java, Python, Spring Boot, MySQL, Git, RESTful...             1Y1S  1.0   
1  SQL, Python, Pandas, Power BI, Tableau, Excel ...             1Y2S  1.3   
2  HTML5, CSS3, JavaScript ES6+, React.js, Tailwi...             2Y1S  2.1   
3  Java, Spring Boot, PostgreSQL, REST APIs, Mav

Clean the Dataset

In [3]:
# Remove unwanted "Unnamed" columns
# When saving from Excel, extra empty columns appear
# We remove them to avoid errors
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
print("\nColumns after removing 'Unnamed':")
print(df.columns)



Columns after removing 'Unnamed':
Index(['Gender', 'District_Location', 'Education_Stream', 'Soft_Skills',
       'Key_Skils', 'Current_semester', 'GPA', 'Experties_Preferred_GPA',
       'Experties_preferred_tech_skills', 'Experties_preferred_Skills',
       'Experties_Preferred_Career_Field', 'English_score', 'Ocean_Openness',
       'Ocean_Conscientiousness', 'Ocean_Extraversion', 'Ocean_Agreeableness',
       'Ocean_Neuroticism', 'Riasec_Realistic', 'Riasec_Investigative',
       'Riasec_Artistic', 'Riasec_Social', 'Riasec_Enterprising',
       'Riasec_Conventional', 'Learning_Style', 'Course_Recommendation',
       'University_Recommendation', 'Skill_Gap_Score', 'Feedback_Score',
       'Data_Consent', 'Timestamp'],
      dtype='object')


Define input features and target

In [4]:
#These are selected based on your research
# Target = career field suggested by experts
# Features = student skill + GPA data
target_col = "Experties_Preferred_Career_Field"
feature_cols = ["Soft_Skills", "Key_Skils", "Current_semester", "GPA"]


Drop rows where target is missing

In [5]:
# If the target is missing, the model cannot learn
# We remove such rows
df = df.dropna(subset=[target_col])
print("\nRows after dropping missing target:", df.shape[0])



Rows after dropping missing target: 1500


Fix missing values in features

In [6]:
# Text columns must not contain NaN → replace with empty string
# GPA must be numeric
# Semester stored as string for encoding

# Fill missing text with empty string
df["Soft_Skills"] = df["Soft_Skills"].fillna("")
df["Key_Skils"] = df["Key_Skils"].fillna("")

# Make sure GPA is numeric
df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce")

# Drop rows where GPA is still NaN after conversion
df = df.dropna(subset=["GPA"])

# Ensure Current_semester is string
df["Current_semester"] = df["Current_semester"].astype(str)

print("\nDtypes after cleaning:")
print(df[feature_cols + [target_col]].dtypes)



Dtypes after cleaning:
Soft_Skills                          object
Key_Skils                            object
Current_semester                     object
GPA                                 float64
Experties_Preferred_Career_Field     object
dtype: object


Create X (features) and y (target)

In [7]:
X = df[feature_cols].copy()
y = df[target_col].copy()

print("\nSample features:")
print(X.head())

print("\nSample target:")
print(y.head())



Sample features:
                                         Soft_Skills  \
0           Communication, Teamwork, Time Management   
1     Leadership, Problem Solving, Critical Thinking   
2            Public Speaking, Teamwork, Adaptability   
3  Communication, Presentation Skills, Emotional ...   
4   Time Management, Work Ethic, Attention to Detail   

                                           Key_Skils Current_semester  GPA  
0  Java, Python, Spring Boot, MySQL, Git, RESTful...             1Y1S  1.0  
1  SQL, Python, Pandas, Power BI, Tableau, Excel ...             1Y2S  1.3  
2  HTML5, CSS3, JavaScript ES6+, React.js, Tailwi...             2Y1S  2.1  
3  Java, Spring Boot, PostgreSQL, REST APIs, Mave...             1Y1S  3.4  
4  React.js, Node.js, Express.js, MongoDB, TypeSc...             2Y2S  4.0  

Sample target:
0    Data Science & Analytics
1    Data Science & Analytics
2    Frontend / Web Developer
3    Data Science & Analytics
4     DevOps / Cloud Engineer
Name: Experties_P

Split Dataset

In [8]:
# We divide into training and testing data
# Training = model learns
# Testing = check accuracy
# stratify=y → keeps target class distribution same
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test
    random_state=42,
    stratify=y          # keep class balance
)

print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])



Train size: 1199
Test size: 300


Preprocessing with ColumnTransformer

In [9]:
# Soft skills and key skills are text → TF-IDF converts text to numbers
# Semester is categorical → OneHotEncoder
# GPA is numeric → StandardScaler for logistic regression stability
text_soft_col = "Soft_Skills"
text_key_col = "Key_Skils"
cat_cols = ["Current_semester"]
num_cols = ["GPA"]

preprocessor = ColumnTransformer(
    transformers=[
        # 1D text columns → TfidfVectorizer
        ("soft_tfidf", TfidfVectorizer(), text_soft_col),
        ("key_tfidf", TfidfVectorizer(), text_key_col),

        # Categorical → OneHot
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),

        # Numeric → StandardScaler
        ("num", StandardScaler(), num_cols),
    ]
)


Build Logistic Regression Pipeline

In [10]:
# We combine preprocessing + model into one pipeline
# This guarantees consistent processing during both training and prediction
# multi_class='multinomial' → supports multiple career options
log_reg_model = LogisticRegression(
    max_iter=1000,
    multi_class="multinomial",
    solver="lbfgs"
)

clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("logreg", log_reg_model)
])


Train the Model

In [11]:
# Model learns the relationship between student data and expert career recommendations
clf.fit(X_train, y_train)
print("✅ Model training completed.")


✅ Model training completed.


D:\ResearchMy\Research_model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluate Model Performance

In [12]:
# Accuracy → overall correct predictions
# Classification report → precision, recall, F1
# Confusion matrix → shows where model confuses classes
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Accuracy: 0.3533

Classification Report:
                                  precision    recall  f1-score   support

  AI / Machine Learning Engineer       0.25      0.19      0.22        21
Cybersecurity / Network Engineer       0.33      0.10      0.15        10
        Data Science & Analytics       0.42      0.59      0.49        82
         DevOps / Cloud Engineer       0.35      0.40      0.37        68
        Frontend / Web Developer       0.26      0.32      0.28        38
      IT / Technology Generalist       0.40      0.12      0.18        17
            Mobile App Developer       0.29      0.27      0.28        22
     Software Engineer / Backend       0.50      0.06      0.11        17
                  UI/UX Designer       0.38      0.20      0.26        25

                        accuracy                           0.35       300
                       macro avg       0.35      0.25      0.26       300
                    weighted avg       0.36      0.35      0.33     

Predict Career for a New Student

In [13]:
# Now your model can make real predictions based on student inputs
new_student = pd.DataFrame([{
    "Soft_Skills": "Communication, Teamwork, Problem solving",
    "Key_Skils": "Java, SQL, Web Development",
    "Current_semester": "2Y1S",
    "GPA": 3.20
}])

predicted_career = clf.predict(new_student)[0]
print("\nPredicted Expert Preferred Career Field:", predicted_career)

print(f"\nAccuracy: {acc:.2f}")



Predicted Expert Preferred Career Field: Data Science & Analytics

Accuracy: 0.35


Adding SHAP Algorithem